In [23]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import pandas as pd
import numpy as np
import math
import plotly.express as px

from cryptosense.data import (
    candles_to_dataframe,
    drop_incomplete_candles,
    fetch_ohlcv_history,
    load_processed,
    save_dataset,
    validate_ohlcv,
)

DATA_DIR = Path.cwd().parent / "data"

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
df = load_processed("BTC/USDT", DATA_DIR)

In [8]:
df["simple_return"]=df["close"].pct_change()
df["log_return"]=np.log(df["close"] / df["close"].shift(1))

In [22]:
df[["close", "simple_return", "log_return"]].head()
df["simple_return"].nlargest(5)



date
2017-12-07 00:00:00+00:00    0.225014
2021-02-08 00:00:00+00:00    0.195361
2019-04-02 00:00:00+00:00    0.171968
2019-10-25 00:00:00+00:00    0.167639
2020-03-13 00:00:00+00:00    0.162208
Name: simple_return, dtype: float64

In [15]:
df["log_return"].nsmallest(5)

date
2020-03-12 00:00:00+00:00   -0.502607
2018-01-16 00:00:00+00:00   -0.216880
2017-09-14 00:00:00+00:00   -0.212657
2022-06-13 00:00:00+00:00   -0.166998
2018-02-05 00:00:00+00:00   -0.164980
Name: log_return, dtype: float64

In [21]:
df["log_return"].sum()


np.float64(2.9468816886172564)

In [25]:
print(df["log_return"].mean())
print(df["log_return"].std())
print(df["log_return"].skew())
print(df["log_return"].kurt())


0.0008804546425507189
0.035271922181481105
-0.9551327342716002
16.00611439011819


In [26]:
fig = px.histogram(df, x="log_return", nbins=200,
                   title="BTC/USDT daily log returns")
fig.update_yaxes(type="log")
fig.show()

In [27]:
r = df["log_return"].dropna()
z = (r - r.mean()) / r.std()

rows = []
for k in [2, 3, 4, 5]:
    rows.append({
        "beyond_std": k,
        "actual_days": int((z.abs() > k).sum()),
        "expected_if_normal": round(len(z) * math.erfc(k / math.sqrt(2)), 2),
    })
pd.DataFrame(rows)

,beyond_std,actual_days,expected_if_normal
0,2,190,152.29
1,3,55,9.04
2,4,19,0.21
3,5,5,0.00
